# ARC Prize 2026 — ARC-AGI-2 Solver v2 (Kaggle Notebook)

## Architecture Overview

This notebook implements a **multi-strategy ensemble solver** combining heuristic pattern matching,
LLM code generation, and evolutionary program synthesis to solve ARC-AGI-2 puzzles.

### Key Improvements over v1

| # | Improvement | Description |
|---|------------|-------------|
| 1 | **Evolutionary Program Synthesis** | Mutate/cross-over LLM-generated programs to explore 10+ candidates per task |
| 2 | **Two-Phase LLM Reasoning** | Phase 1: natural-language analysis → Phase 2: Python code generation |
| 3 | **Expanded Heuristic Library** | 30+ solvers: geometric, color, object, scaling, tiling, pattern, logic |
| 4 | **Partial Match Scoring** | Score 0–1 per training pair instead of all-or-nothing |
| 5 | **Multi-Strategy Ensemble** | Fast heuristics first, then LLM, combine with voting |
| 6 | **Better Prompt Engineering** | Color names, grid dimensions, object descriptions in prompts |
| 7 | **DreamCoder Abstraction** | Extract reusable sub-functions from solved tasks |

### Pipeline

```
For each task:
  1. Run 30+ heuristic solvers → pick best by partial-match score
  2. If score < 1.0, run two-phase LLM reasoning (analyze → code)
  3. Evolve LLM candidates via mutation/crossover → 10+ candidates
  4. Ensemble: pick top-2 distinct predictions by confidence
  5. Extract abstractions for future tasks (DreamCoder)
```

### Kaggle Constraints
- GPU: T4/P100 (16 GB), no internet
- Time limit: 9 hours (~81 s per task for 400 eval tasks)
- Output: `submission.json` with 2 attempts per task

In [ ]:
# ============================================================================
# CELL 1: Configuration, Imports & Constants
# ============================================================================
"""
ARC-AGI-2 Solver v2 — Configuration & Imports

Self-contained Kaggle notebook. Uses: transformers, torch, numpy, scipy, tqdm.
No internet access required; Qwen3-4B is pre-cached on Kaggle GPUs.
"""

import json
import os
import copy
import time
import sys
import traceback
import textwrap
import threading
import re
import hashlib
import random
import math
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple, Callable
from itertools import product as itertools_product

import numpy as np

# Lazy imports — only needed for LLM (GPU) path
_HAS_TORCH = False
_HAS_TRANSFORMERS = False
_HAS_BNB = False
try:
    import torch
    _HAS_TORCH = True
except ImportError:
    pass
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    _HAS_TRANSFORMERS = True
except ImportError:
    pass
try:
    import bitsandbytes  # noqa: F401
    _HAS_BNB = True
except ImportError:
    pass

try:
    from tqdm.auto import tqdm
except ImportError:
    # Fallback: no-op progress
    def tqdm(iterable, **kwargs):
        desc = kwargs.get('desc', '')
        total = kwargs.get('total', len(iterable) if hasattr(iterable, '__len__') else None)
        if desc:
            print(f"{desc}...")
        for item in iterable:
            yield item

# --------------------------------------------------------------------------
# Paths (Kaggle)
# --------------------------------------------------------------------------
DATA_DIR = "/kaggle/input/arc-prize-2026-arc-agi-2"
OUTPUT_PATH = "/kaggle/working/submission.json"

# --------------------------------------------------------------------------
# LLM settings
# --------------------------------------------------------------------------
MODEL_NAME = "Qwen/Qwen3-4B"
QUANTIZATION_4BIT = True
MAX_NEW_TOKENS = 2048
MAX_INPUT_TOKENS = 30000

# --------------------------------------------------------------------------
# Solver settings
# --------------------------------------------------------------------------
CODE_TIMEOUT_SECONDS = 10       # per code execution
MAX_LLM_TIME_SECONDS = 300      # max wall-clock for LLM attempts per task
NUM_SUBMISSION_ATTEMPTS = 2     # attempts per task in submission
EVOLUTIONARY_POPULATION = 12    # candidates in evolutionary pool
EVOLUTIONARY_GENERATIONS = 3    # mutation/crossover rounds

# --------------------------------------------------------------------------
# ARC color palette (for prompt engineering)
# --------------------------------------------------------------------------
COLOR_NAMES = {
    0: "black", 1: "blue", 2: "red", 3: "green", 4: "yellow",
    5: "gray", 6: "magenta", 7: "orange", 8: "cyan", 9: "brown",
}
COLOR_NAME_TO_ID = {v: k for k, v in COLOR_NAMES.items()}

# --------------------------------------------------------------------------
# DreamCoder shared abstraction library
# --------------------------------------------------------------------------
ABSTRACTION_LIBRARY: Dict[str, str] = {}  # name → code string

print(f"[CONFIG] Python {sys.version}")
print(f"[CONFIG] Torch available: {_HAS_TORCH}")
print(f"[CONFIG] Transformers available: {_HAS_TRANSFORMERS}")
print(f"[CONFIG] BitsAndBytes available: {_HAS_BNB}")
if _HAS_TORCH:
    print(f"[CONFIG] CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"[CONFIG] GPU: {torch.cuda.get_device_name(0)}")
print(f"[CONFIG] Model: {MODEL_NAME}")
print(f"[CONFIG] NumPy: {np.__version__}")

## Grid Utility Functions

In [ ]:
# ============================================================================
# CELL 2: Grid Utility Functions
# ============================================================================
"""
Core grid manipulation utilities used throughout the solver.
All functions accept and return list-of-lists (not numpy arrays) for
compatibility with ARC JSON format and exec-based code generation.
"""

def grid_to_str(grid: List[List[int]], label: str = "") -> str:
    """Convert grid to multi-line string with optional label and dimensions."""
    if not grid:
        return f"{label}: (empty)"
    rows = len(grid)
    cols = len(grid[0]) if grid else 0
    header = f"{label} ({rows}x{cols}):" if label else f"({rows}x{cols}):"
    lines = [header]
    for row in grid:
        lines.append(" ".join(str(v) for v in row))
    return "\n".join(lines)


def grids_equal(a: List[List[int]], b: List[List[int]]) -> bool:
    """Check exact equality of two grids."""
    if len(a) != len(b):
        return False
    for ra, rb in zip(a, b):
        if len(ra) != len(rb):
            return False
        for va, vb in zip(ra, rb):
            if va != vb:
                return False
    return True


def grid_to_numpy(grid: List[List[int]]) -> np.ndarray:
    return np.array(grid, dtype=np.int64)


def numpy_to_grid(arr: np.ndarray) -> List[List[int]]:
    return arr.astype(int).tolist()


def normalize_grid(result: Any) -> Optional[List[List[int]]]:
    """Coerce arbitrary return value into a valid grid (list-of-lists of ints 0–9)."""
    if isinstance(result, list):
        if all(isinstance(row, list) for row in result):
            if all(isinstance(v, (int, np.integer)) for row in result for v in row):
                return [[int(v) % 10 for v in row] for row in result]
        if all(isinstance(row, (list, np.ndarray)) for row in result):
            try:
                return [[int(v) % 10 for v in row] for row in result]
            except (TypeError, ValueError):
                pass
    if isinstance(result, np.ndarray):
        if result.ndim == 2:
            return result.astype(int).tolist()
        elif result.ndim == 1:
            return [result.astype(int).tolist()]
    if isinstance(result, tuple):
        return normalize_grid(list(result))
    return None


def ensure_grid(result: Any) -> List[List[int]]:
    """Like normalize_grid but returns [[0]] on failure instead of None."""
    g = normalize_grid(result)
    return g if g is not None else [[0]]


def grid_hash(grid: List[List[int]]) -> str:
    """Deterministic hash of a grid for deduplication."""
    s = json.dumps(grid, sort_keys=True)
    return hashlib.md5(s.encode()).hexdigest()[:12]


def grid_shape(grid: List[List[int]]) -> Tuple[int, int]:
    if not grid:
        return (0, 0)
    return (len(grid), len(grid[0]))


# --------------------------------------------------------------------------
# Component analysis (flood fill)
# --------------------------------------------------------------------------
def find_components(grid: List[List[int]], connectivity: int = 4) -> List[List[Tuple[int, int]]]:
    """
    Find connected components of non-zero cells using flood fill.
    connectivity: 4 (up/down/left/right) or 8 (includes diagonals).
    Returns list of components, each a list of (row, col) tuples.
    """
    arr = grid_to_numpy(grid)
    h, w = arr.shape
    visited = np.zeros((h, w), dtype=bool)
    components = []

    if connectivity == 4:
        neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    else:
        neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1),
                     (-1, -1), (-1, 1), (1, -1), (1, 1)]

    for r in range(h):
        for c in range(w):
            if arr[r, c] != 0 and not visited[r, c]:
                color = int(arr[r, c])
                component = []
                stack = [(r, c)]
                while stack:
                    cr, cc = stack.pop()
                    if cr < 0 or cr >= h or cc < 0 or cc >= w:
                        continue
                    if visited[cr, cc]:
                        continue
                    if arr[cr, cc] != color:
                        continue
                    visited[cr, cc] = True
                    component.append((cr, cc))
                    for dr, dc in neighbors:
                        stack.append((cr + dr, cc + dc))
                components.append(component)

    return components


def component_bounding_box(component: List[Tuple[int, int]]) -> Tuple[int, int, int, int]:
    """Return (min_r, min_c, max_r, max_c) for a component."""
    rows = [p[0] for p in component]
    cols = [p[1] for p in component]
    return (min(rows), min(cols), max(rows), max(cols))


def extract_component_grid(grid: List[List[int]], component: List[Tuple[int, int]]) -> List[List[int]]:
    """Extract a component from the grid into its own cropped sub-grid."""
    min_r, min_c, max_r, max_c = component_bounding_box(component)
    comp_set = set(component)
    arr = grid_to_numpy(grid)
    return arr[min_r:max_r+1, min_c:max_c+1].tolist()


def count_colors(grid: List[List[int]]) -> Counter:
    """Count occurrences of each color in the grid."""
    arr = grid_to_numpy(grid)
    flat = arr.flatten().tolist()
    return Counter(int(v) for v in flat)


# --------------------------------------------------------------------------
# Grid feature extraction (for conditional logic & prompt engineering)
# --------------------------------------------------------------------------
def extract_grid_features(grid: List[List[int]]) -> Dict[str, Any]:
    """
    Extract comprehensive features from a grid for conditional routing
    and LLM prompt enrichment.
    """
    arr = grid_to_numpy(grid)
    h, w = arr.shape
    colors = count_colors(grid)
    nonzero_colors = {c for c in colors if c != 0}
    total_cells = h * w
    nonzero_cells = sum(colors[c] for c in nonzero_colors)
    density = nonzero_cells / max(total_cells, 1)
    comps_4 = find_components(grid, 4)
    comps_8 = find_components(grid, 8)

    return {
        'height': h, 'width': w, 'is_square': h == w,
        'wider_than_tall': w > h, 'taller_than_wide': h > w,
        'size_parity': (h * w) % 2,
        'colors': dict(colors),
        'nonzero_colors': nonzero_colors,
        'num_colors': len(nonzero_colors),
        'dominant_color': colors.most_common(1)[0][0] if colors else 0,
        'density': density,
        'num_components_4': len(comps_4),
        'num_components_8': len(comps_8),
        'nonzero_cells': nonzero_cells,
        'sym_h': bool(np.array_equal(arr, arr[:, ::-1])),
        'sym_v': bool(np.array_equal(arr, arr[::-1, :])),
    }


def describe_grid_natural_language(grid: List[List[int]], label: str = "Grid") -> str:
    """
    Generate a natural-language description of the grid for LLM prompts.
    Includes dimensions, colors, objects, and spatial layout.
    """
    features = extract_grid_features(grid)
    parts = [f"{label}: {features['height']} rows x {features['width']} cols"]

    # Color description
    color_descs = []
    for c in sorted(features['nonzero_colors']):
        name = COLOR_NAMES.get(c, str(c))
        count = features['colors'][c]
        color_descs.append(f"{name}({c}) x{count}")
    if color_descs:
        parts.append(f"Colors: {', '.join(color_descs)}")
    else:
        parts.append("Colors: all black (empty)")

    # Object description
    n_comp = features['num_components_4']
    if n_comp > 0:
        parts.append(f"Objects: {n_comp} connected component(s) (4-conn)")
    else:
        parts.append("Objects: none")

    # Shape
    shape = "square" if features['is_square'] else (
        "wide" if features['wider_than_tall'] else "tall")
    parts.append(f"Shape: {shape}")

    # Symmetry
    syms = []
    if features['sym_h']:
        syms.append("horizontal")
    if features['sym_v']:
        syms.append("vertical")
    if syms:
        parts.append(f"Symmetry: {', '.join(syms)}")

    return " | ".join(parts)


print("[UTILS] Grid utility functions loaded.")
print(f"[UTILS] Color palette: {COLOR_NAMES}")

## Data Loading

In [ ]:
# ============================================================================
# CELL 3: Data Loading
# ============================================================================

def load_task(filepath: str) -> Dict[str, Any]:
    """Load a single ARC task JSON file."""
    with open(filepath, "r") as f:
        return json.load(f)


def load_all_tasks(data_dir: str, split: str = "evaluation") -> Dict[str, Dict]:
    """Load every task in a split directory."""
    split_dir = os.path.join(data_dir, "data", split)
    tasks = {}
    if not os.path.isdir(split_dir):
        print(f"[WARN] Directory not found: {split_dir}")
        return tasks
    for fname in sorted(os.listdir(split_dir)):
        if fname.endswith(".json"):
            task_id = fname.replace(".json", "")
            tasks[task_id] = load_task(os.path.join(split_dir, fname))
    return tasks


def load_arc_data(data_dir: str) -> Tuple[Dict[str, Dict], Dict[str, Dict]]:
    """Load both evaluation and training splits. Returns (eval_tasks, train_tasks)."""
    eval_tasks = load_all_tasks(data_dir, "evaluation")
    train_tasks = load_all_tasks(data_dir, "training")
    print(f"[DATA] Loaded {len(eval_tasks)} evaluation, {len(train_tasks)} training tasks.")
    return eval_tasks, train_tasks


def print_task_summary(task: Dict, task_id: str = "") -> None:
    """Print a concise summary of a task."""
    n_train = len(task.get('train', []))
    n_test = len(task.get('test', []))
    if task_id:
        print(f"  Task {task_id}: {n_train} train pairs, {n_test} test inputs")
    for i, pair in enumerate(task.get('train', [])):
        inp_s = grid_shape(pair['input'])
        out_s = grid_shape(pair['output'])
        print(f"    Train {i+1}: {inp_s} → {out_s}")
    for i, t in enumerate(task.get('test', [])):
        inp_s = grid_shape(t['input'])
        print(f"    Test {i+1}: input {inp_s}")


print("[DATA] Data loading functions ready.")

## Expanded Heuristic Library (30+ Solvers)

In [ ]:
# ============================================================================
# CELL 4: Expanded Heuristic Library — Geometric & Color Solvers
# ============================================================================
"""
30+ heuristic solvers organized by category.
Each solver is a function: grid → grid (or raises on failure).
They are combined with partial-match scoring to find the best match.
"""

# === GEOMETRIC SOLVERS (1–8) =========================================

def h_identity(grid):
    return copy.deepcopy(grid)

def h_rotate90(grid):
    return np.rot90(grid_to_numpy(grid), 1).tolist()

def h_rotate180(grid):
    return np.rot90(grid_to_numpy(grid), 2).tolist()

def h_rotate270(grid):
    return np.rot90(grid_to_numpy(grid), 3).tolist()

def h_flip_h(grid):
    g = grid_to_numpy(grid)
    return g[:, ::-1].tolist()

def h_flip_v(grid):
    g = grid_to_numpy(grid)
    return g[::-1, :].tolist()

def h_flip_diag(grid):
    """Transpose (flip along main diagonal)."""
    g = grid_to_numpy(grid)
    return g.T.tolist()

def h_flip_anti_diag(grid):
    """Flip along anti-diagonal."""
    g = grid_to_numpy(grid)
    return np.rot90(g.T).tolist()


# === COLOR SOLVERS (9–16) ==============================================

def h_color_complement(grid):
    """Map each color c to 9-c (complement)."""
    g = grid_to_numpy(grid)
    return (9 - g).tolist()

def h_color_add_k(grid, k=1):
    """Map each color c to (c+k) mod 10."""
    g = grid_to_numpy(grid)
    return ((g + k) % 10).tolist()

def h_color_sub_k(grid, k=1):
    """Map each color c to (c-k) mod 10."""
    g = grid_to_numpy(grid)
    return ((g - k) % 10).tolist()

def h_fill_most_common(grid):
    """Replace all non-zero with most common non-zero color."""
    g = grid_to_numpy(grid)
    vals = g[g > 0].tolist()
    if not vals:
        return copy.deepcopy(grid)
    mc = Counter(int(v) for v in vals).most_common(1)[0][0]
    return np.where(g > 0, mc, 0).tolist()

def h_extract_color(grid, color=1):
    """Keep only one color, set rest to 0."""
    g = grid_to_numpy(grid)
    return np.where(g == color, color, 0).tolist()

def h_remap_colors(grid):
    """Remap colors to 1..N in order of first appearance."""
    g = grid_to_numpy(grid)
    unique = []
    mapping = {}
    for val in g.flatten():
        v = int(val)
        if v not in mapping:
            if v == 0:
                mapping[v] = 0
            else:
                mapping[v] = len(unique) + 1
                unique.append(v)
    result = np.zeros_like(g)
    for v, new_v in mapping.items():
        result[g == v] = new_v
    return result.tolist()

def h_count_colors_output(grid):
    """Output: single row with count of each non-zero color."""
    g = grid_to_numpy(grid)
    colors = sorted(set(int(c) for c in g.flatten() if c != 0))
    if not colors:
        return [[0]]
    counts = [int(np.sum(g == c)) for c in colors]
    return [counts]

def h_invert_colors(grid):
    """Cycle non-zero colors forward by one position."""
    g = grid_to_numpy(grid)
    unique = sorted(set(int(c) for c in g.flatten() if c != 0))
    if len(unique) <= 1:
        return copy.deepcopy(grid)
    mapping = {0: 0}
    for i, v in enumerate(unique):
        mapping[v] = unique[(i + 1) % len(unique)]
    result = np.zeros_like(g)
    for old, new in mapping.items():
        result[g == old] = new
    return result.tolist()


# === OBJECT SOLVERS (17–22) ============================================

def h_crop(grid):
    """Remove all-zero border rows and columns."""
    g = grid_to_numpy(grid)
    if g.size == 0:
        return grid
    rows_nz = np.where(g.any(axis=1))[0]
    cols_nz = np.where(g.any(axis=0))[0]
    if len(rows_nz) == 0 or len(cols_nz) == 0:
        return [[0]]
    return g[rows_nz[0]:rows_nz[-1]+1, cols_nz[0]:cols_nz[-1]+1].tolist()

def h_crop_each_color(grid):
    """Crop each color's bounding box separately, stack horizontally."""
    g = grid_to_numpy(grid)
    colors = sorted(set(int(c) for c in g.flatten() if c != 0))
    if not colors:
        return [[0]]
    sub_grids = []
    max_h = 0
    for c in colors:
        mask = (g == c).astype(int)
        rows_nz = np.where(mask.any(axis=1))[0]
        cols_nz = np.where(mask.any(axis=0))[0]
        if len(rows_nz) == 0:
            continue
        sub = mask[rows_nz[0]:rows_nz[-1]+1, cols_nz[0]:cols_nz[-1]+1]
        sub_grids.append(sub)
        max_h = max(max_h, sub.shape[0])
    if not sub_grids:
        return [[0]]
    # Pad to same height and concatenate
    padded = []
    for sg in sub_grids:
        if sg.shape[0] < max_h:
            pad = np.zeros((max_h - sg.shape[0], sg.shape[1]), dtype=int)
            sg = np.vstack([sg, pad])
        padded.append(sg)
    return np.hstack(padded).tolist()

def h_extract_largest_component(grid):
    """Keep only the largest connected component."""
    comps = find_components(grid, 4)
    if not comps:
        return [[0] * len(grid[0]) for _ in range(len(grid))]
    largest = max(comps, key=len)
    g = np.zeros_like(grid_to_numpy(grid))
    color = int(grid_to_numpy(grid)[largest[0][0], largest[0][1]])
    for r, c in largest:
        g[r, c] = color
    return g.tolist()

def h_extract_smallest_component(grid):
    """Keep only the smallest connected component."""
    comps = find_components(grid, 4)
    if not comps:
        return [[0] * len(grid[0]) for _ in range(len(grid))]
    smallest = min(comps, key=len)
    g = np.zeros_like(grid_to_numpy(grid))
    color = int(grid_to_numpy(grid)[smallest[0][0], smallest[0][1]])
    for r, c in smallest:
        g[r, c] = color
    return g.tolist()

def h_move_to_topleft(grid):
    """Crop content and place in top-left corner (same as crop)."""
    return h_crop(grid)

def h_extract_all_components_separate(grid):
    """Extract each component, return largest one cropped."""
    return h_extract_largest_component(grid)


# === SCALING SOLVERS (23–26) ===========================================

def h_scale_2x(grid):
    """Scale up by 2x using nearest-neighbor."""
    g = grid_to_numpy(grid)
    return np.repeat(np.repeat(g, 2, axis=0), 2, axis=1).tolist()

def h_scale_3x(grid):
    """Scale up by 3x using nearest-neighbor."""
    g = grid_to_numpy(grid)
    return np.repeat(np.repeat(g, 3, axis=0), 3, axis=1).tolist()

def h_shrink_by_half(grid):
    """Shrink by taking every other row and column."""
    g = grid_to_numpy(grid)
    if g.shape[0] < 2 or g.shape[1] < 2:
        return copy.deepcopy(grid)
    return g[::2, ::2].tolist()

def h_shrink_by_third(grid):
    """Shrink by taking every third row and column."""
    g = grid_to_numpy(grid)
    if g.shape[0] < 3 or g.shape[1] < 3:
        return copy.deepcopy(grid)
    return g[::3, ::3].tolist()


# === TILING SOLVERS (27–30) ============================================

def h_tile_2x2(grid):
    """Tile the grid into a 2x2 arrangement."""
    g = grid_to_numpy(grid)
    return np.tile(g, (2, 2)).tolist()

def h_tile_3x3(grid):
    """Tile the grid into a 3x3 arrangement."""
    g = grid_to_numpy(grid)
    return np.tile(g, (3, 3)).tolist()

def h_repeat_h(grid):
    """Repeat grid horizontally twice."""
    g = grid_to_numpy(grid)
    return np.tile(g, (1, 2)).tolist()

def h_repeat_v(grid):
    """Repeat grid vertically twice."""
    g = grid_to_numpy(grid)
    return np.tile(g, (2, 1)).tolist()


# === PATTERN SOLVERS (31–34) ===========================================

def h_mirror_h_full(grid):
    """Concatenate grid with its horizontal mirror."""
    g = grid_to_numpy(grid)
    return np.hstack([g, g[:, ::-1]]).tolist()

def h_mirror_v_full(grid):
    """Concatenate grid with its vertical mirror."""
    g = grid_to_numpy(grid)
    return np.vstack([g, g[::-1, :]]).tolist()

def h_mirror_4way(grid):
    """Create 4-way symmetry: horizontal + vertical mirror."""
    g = grid_to_numpy(grid)
    top = np.hstack([g, g[:, ::-1]])
    return np.vstack([top, top[::-1, :]]).tolist()

def h_extract_border(grid):
    """Extract only the border pixels, set interior to 0."""
    g = grid_to_numpy(grid).copy()
    if g.shape[0] > 2 and g.shape[1] > 2:
        g[1:-1, 1:-1] = 0
    return g.tolist()


# === LOGIC / CONDITIONAL SOLVERS (35–38) ===============================

def h_conditional_color_density(grid):
    """If density > 0.5, return complement; else return as-is."""
    g = grid_to_numpy(grid)
    density = np.sum(g != 0) / max(g.size, 1)
    if density > 0.5:
        return (9 - g).tolist()
    return copy.deepcopy(grid)

def h_conditional_size_rotate(grid):
    """If grid is square, rotate 90; if wider, flip h; if taller, flip v."""
    g = grid_to_numpy(grid)
    h, w = g.shape
    if h == w:
        return np.rot90(g, 1).tolist()
    elif w > h:
        return g[:, ::-1].tolist()
    else:
        return g[::-1, :].tolist()

def h_threshold_binary(grid):
    """Convert to binary: non-zero → 1, zero stays 0."""
    g = grid_to_numpy(grid)
    return (g > 0).astype(int).tolist()

def h_replace_zero_with_dominant(grid):
    """Replace 0 with the most common color."""
    g = grid_to_numpy(grid)
    colors = Counter(int(v) for v in g.flatten())
    if colors:
        dominant = colors.most_common(1)[0][0]
        return np.where(g == 0, dominant, g).tolist()
    return copy.deepcopy(grid)


print(f"[HEURISTICS] {38} heuristic solvers defined.")

## Heuristic Registration & Partial Match Scoring

In [ ]:
# ============================================================================
# CELL 5: Heuristic Registration, Color-Parameterized Solvers & Partial Match
# ============================================================================

# --- Register all base solvers -------------------------------------------
BASE_HEURISTICS: List[Tuple[str, Callable]] = [
    # Geometric
    ("identity",          h_identity),
    ("rotate90",          h_rotate90),
    ("rotate180",         h_rotate180),
    ("rotate270",         h_rotate270),
    ("flip_h",            h_flip_h),
    ("flip_v",            h_flip_v),
    ("flip_diag",         h_flip_diag),
    ("flip_anti_diag",    h_flip_anti_diag),
    # Color
    ("color_complement",  h_color_complement),
    ("invert_colors",     h_invert_colors),
    ("fill_most_common",  h_fill_most_common),
    ("remap_colors",      h_remap_colors),
    ("count_colors_out",  h_count_colors_output),
    # Object
    ("crop",              h_crop),
    ("crop_each_color",   h_crop_each_color),
    ("largest_comp",      h_extract_largest_component),
    ("smallest_comp",     h_extract_smallest_component),
    # Scaling
    ("scale_2x",          h_scale_2x),
    ("scale_3x",          h_scale_3x),
    ("shrink_half",       h_shrink_by_half),
    ("shrink_third",      h_shrink_by_third),
    # Tiling
    ("tile_2x2",          h_tile_2x2),
    ("tile_3x3",          h_tile_3x3),
    ("repeat_h",          h_repeat_h),
    ("repeat_v",          h_repeat_v),
    # Pattern
    ("mirror_h_full",     h_mirror_h_full),
    ("mirror_v_full",     h_mirror_v_full),
    ("mirror_4way",       h_mirror_4way),
    ("extract_border",    h_extract_border),
    # Logic
    ("threshold_binary",  h_threshold_binary),
    ("replace_zero_dom",  h_replace_zero_with_dominant),
]


# --- Parameterized color solvers (expand to 10+ more) ---------------------
def _make_extract_color_fn(color):
    def fn(grid):
        return h_extract_color(grid, color)
    return fn

def _make_color_add_fn(k):
    def fn(grid):
        return h_color_add_k(grid, k)
    return fn

def _make_color_sub_fn(k):
    def fn(grid):
        return h_color_sub_k(grid, k)
    return fn

def _make_conditional_rotate_fn(angle):
    def fn(grid):
        return np.rot90(grid_to_numpy(grid), angle).tolist()
    return fn

for c in range(1, 10):
    BASE_HEURISTICS.append((f"extract_color_{c}", _make_extract_color_fn(c)))
for k in range(1, 10):
    BASE_HEURISTICS.append((f"color_add_{k}", _make_color_add_fn(k)))
    BASE_HEURISTICS.append((f"color_sub_{k}", _make_color_sub_fn(k)))

# --- Composite solvers (chain two operations) -----------------------------
COMPOSITE_HEURISTICS: List[Tuple[str, Callable]] = []

# Common composites that appear in ARC tasks
COMPOSITE_DEFS = [
    ("crop_rotate90",   [h_crop, h_rotate90]),
    ("crop_rotate180",  [h_crop, h_rotate180]),
    ("crop_flip_h",     [h_crop, h_flip_h]),
    ("crop_flip_v",     [h_crop, h_flip_v]),
    ("crop_scale2x",    [h_crop, h_scale_2x]),
    ("largest_comp_crop", [h_extract_largest_component, h_crop]),
    ("crop_mirror_h",   [h_crop, h_mirror_h_full]),
    ("crop_mirror_v",   [h_crop, h_mirror_v_full]),
    ("rotate90_crop",   [h_rotate90, h_crop]),
    ("flip_h_crop",     [h_flip_h, h_crop]),
    ("flip_v_crop",     [h_flip_v, h_crop]),
    ("threshold_crop",  [h_threshold_binary, h_crop]),
    ("remap_crop",      [h_remap_colors, h_crop]),
    ("crop_extract_color_1", [h_crop, _make_extract_color_fn(1)]),
    ("crop_extract_color_2", [h_crop, _make_extract_color_fn(2)]),
    ("crop_extract_color_3", [h_crop, _make_extract_color_fn(3)]),
]

for name, fns in COMPOSITE_DEFS:
    def make_composite(funcs):
        def fn(grid):
            result = copy.deepcopy(grid)
            for f in funcs:
                result = normalize_grid(f(result))
                if result is None:
                    return None
            return result
        return fn
    COMPOSITE_HEURISTICS.append((name, make_composite(fns)))

ALL_HEURISTICS = BASE_HEURISTICS + COMPOSITE_HEURISTICS


# === PARTIAL MATCH SCORING =================================================

def score_heuristic_on_training(
    heuristic_fn: Callable,
    train_pairs: List[Dict],
) -> Tuple[float, int, int]:
    """
    Score a heuristic against training pairs.
    Returns (score, num_passed, total).
    score = num_passed / total (0.0 to 1.0)
    """
    passed = 0
    total = len(train_pairs)
    for pair in train_pairs:
        try:
            result = heuristic_fn(pair["input"])
            norm = normalize_grid(result)
            if norm is not None and grids_equal(norm, pair["output"]):
                passed += 1
        except Exception:
            continue
    return (passed / total if total > 0 else 0.0, passed, total)


def find_best_heuristics(
    train_pairs: List[Dict],
    min_score: float = 1.0,
    max_results: int = 5,
) -> List[Tuple[str, Callable, float, int, int]]:
    """
    Find all heuristics that score >= min_score on training pairs.
    Returns list of (name, fn, score, passed, total) sorted by score desc.
    """
    results = []
    for name, fn in ALL_HEURISTICS:
        score, passed, total = score_heuristic_on_training(fn, train_pairs)
        if score >= min_score:
            results.append((name, fn, score, passed, total))
    results.sort(key=lambda x: (-x[2], -x[3], x[0]))
    return results[:max_results]


def find_best_partial_heuristics(
    train_pairs: List[Dict],
    max_results: int = 10,
) -> List[Tuple[str, Callable, float, int, int]]:
    """
    Find heuristics sorted by partial-match score (including imperfect ones).
    Useful for ensemble voting and identifying promising approaches.
    """
    results = []
    for name, fn in ALL_HEURISTICS:
        score, passed, total = score_heuristic_on_training(fn, train_pairs)
        if score > 0.0:  # at least some pairs correct
            results.append((name, fn, score, passed, total))
    results.sort(key=lambda x: (-x[2], -x[3], x[0]))
    return results[:max_results]


print(f"[HEURISTICS] Total registered: {len(ALL_HEURISTICS)} "
      f"({len(BASE_HEURISTICS)} base + {len(COMPOSITE_HEURISTICS)} composite)")

## Safe Code Execution & Verification

In [ ]:
# ============================================================================
# CELL 6: Safe Code Execution & Verification
# ============================================================================
"""
Sandboxed code execution with timeout and partial-match scoring.
All LLM-generated code is executed here with safety guarantees.
"""

def _run_code_in_thread(code, input_grid, result_container, error_container):
    """Worker thread for code execution."""
    try:
        local_ns = {"np": np, "copy": copy, "math": math,
                     "Counter": Counter, "defaultdict": defaultdict,
                     "set": set, "sorted": sorted, "len": len, "range": range,
                     "enumerate": enumerate, "zip": zip, "list": list,
                     "int": int, "float": float, "bool": bool, "max": max, "min": min,
                     "abs": abs, "sum": sum, "any": any, "all": all,
                     "print": lambda *a, **k: None,  # suppress prints
                     "__builtins__": __builtins__}
        exec(code, local_ns)
        transform_fn = local_ns.get("transform")
        if transform_fn is None:
            error_container.append("No 'transform' function defined.")
            return
        result = transform_fn(copy.deepcopy(input_grid))
        result_container.append(result)
    except Exception as e:
        error_container.append(str(e))


def safe_execute_code(
    code: str,
    input_grid: List[List[int]],
    timeout: float = CODE_TIMEOUT_SECONDS,
) -> Optional[List[List[int]]]:
    """Execute code safely with timeout. Returns output grid or None."""
    result_container = []
    error_container = []
    thread = threading.Thread(
        target=_run_code_in_thread,
        args=(code, input_grid, result_container, error_container),
        daemon=True,
    )
    thread.start()
    thread.join(timeout=timeout)

    if thread.is_alive() or error_container or not result_container:
        return None

    return normalize_grid(result_container[0])


def verify_code_on_training(
    code: str,
    train_pairs: List[Dict],
) -> Tuple[float, int, int]:
    """
    Verify code against training pairs with PARTIAL MATCH scoring.
    Returns (score, num_passed, total).
    score = num_passed / total (0.0 to 1.0)
    """
    passed = 0
    total = len(train_pairs)
    for pair in train_pairs:
        result = safe_execute_code(code, pair["input"])
        if result is not None and grids_equal(result, pair["output"]):
            passed += 1
    return (passed / total if total > 0 else 0.0, passed, total)


def extract_function_code(llm_output: str) -> Optional[str]:
    """Extract transform() function from raw LLM text output."""
    if not llm_output:
        return None

    # Strategy 1: fenced code block
    fence_match = re.search(r"```(?:python)?\s*\n?(.*?)```", llm_output, re.DOTALL)
    if fence_match:
        code_block = fence_match.group(1).strip()
        if "def transform" in code_block:
            fn_match = re.search(
                r"(def\s+transform\s*\(\s*input_grid\s*\)\s*:.*?)"
                r"(?=\n(?:def |class |$))",
                code_block, re.DOTALL
            )
            if fn_match:
                return fn_match.group(1).strip()
            return code_block

    # Strategy 2: direct def transform
    fn_match = re.search(
        r"(def\s+transform\s*\(\s*input_grid\s*\)\s*:.*?)"
        r"(?=\n(?:def |class |$))",
        llm_output, re.DOTALL
    )
    if fn_match:
        return fn_match.group(1).strip()

    # Strategy 3: find first def transform
    idx = llm_output.find("def transform")
    if idx >= 0:
        return llm_output[idx:].strip()

    return None


print("[EXEC] Safe code execution and verification ready.")

## DreamCoder-Style Abstraction Library

In [ ]:
# ============================================================================
# CELL 7: DreamCoder-Style Abstraction Library
# ============================================================================
"""
Maintains a growing library of reusable sub-functions extracted from
successfully solved tasks. These are injected into future LLM prompts
as building blocks, improving code generation accuracy.

The key insight (from DreamCoder, Ellis et al. 2021) is that programs
can be decomposed into reusable abstractions. By maintaining a library
of verified sub-programs, the LLM can compose more complex solutions.
"""

class AbstractionLibrary:
    """
    Manages a library of reusable code abstractions.
    Each abstraction has: name, code, usage_count, categories.
    """

    def __init__(self):
        self.abstractions: Dict[str, Dict] = {}  # name → {code, uses, categories}
        self._init_builtin_abstractions()

    def _init_builtin_abstractions(self):
        """Seed the library with common ARC utility functions."""
        builtins = {
            "find_bounding_box": (
                "def find_bounding_box(grid):\n"
                "    rows = [i for i, row in enumerate(grid) if any(c != 0 for c in row)]\n"
                "    cols = [j for j in range(len(grid[0])) if any(grid[i][j] != 0 for i in range(len(grid)))]\n"
                "    if not rows or not cols: return (0, 0, 0, 0)\n"
                "    return (min(rows), min(cols), max(rows), max(cols))\n",
                ["geometry", "crop"]
            ),
            "crop_grid": (
                "def crop_grid(grid):\n"
                "    import numpy as np\n"
                "    g = np.array(grid)\n"
                "    r = np.where(g.any(axis=1))[0]\n"
                "    c = np.where(g.any(axis=0))[0]\n"
                "    if len(r) == 0: return [[0]]\n"
                "    return g[r[0]:r[-1]+1, c[0]:c[-1]+1].tolist()\n",
                ["crop", "geometry"]
            ),
            "find_objects": (
                "def find_objects(grid, connectivity=4):\n"
                "    import numpy as np\n"
                "    g = np.array(grid)\n"
                "    visited = np.zeros_like(g, dtype=bool)\n"
                "    objects = []\n"
                "    neighbors = [(-1,0),(1,0),(0,-1),(0,1)] if connectivity==4 else [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]\n"
                "    for i in range(g.shape[0]):\n"
                "        for j in range(g.shape[1]):\n"
                "            if g[i,j] != 0 and not visited[i,j]:\n"
                "                color = int(g[i,j])\n"
                "                obj = []\n"
                "                stack = [(i,j)]\n"
                "                while stack:\n"
                "                    ci,cj = stack.pop()\n"
                "                    if ci<0 or ci>=g.shape[0] or cj<0 or cj>=g.shape[1] or visited[ci,cj] or g[ci,cj]!=color: continue\n"
                "                    visited[ci,cj] = True\n"
                "                    obj.append((ci,cj))\n"
                "                    for di,dj in neighbors: stack.append((ci+di,cj+dj))\n"
                "                objects.append(obj)\n"
                "    return objects\n",
                ["objects", "perception"]
            ),
            "get_color_counts": (
                "def get_color_counts(grid):\n"
                "    counts = {}\n"
                "    for row in grid:\n"
                "        for v in row:\n"
                "            counts[v] = counts.get(v, 0) + 1\n"
                "    return counts\n",
                ["color", "counting"]
            ),
            "rotate_grid": (
                "def rotate_grid(grid, times=1):\n"
                "    import numpy as np\n"

                "    g = np.array(grid)\n"
                "    return np.rot90(g, times).tolist()\n",
                ["geometry", "rotation"]
            ),
            "check_symmetry": (
                "def check_symmetry(grid):\n"
                "    import numpy as np\n"
                "    g = np.array(grid)\n"
                "    h_sym = np.array_equal(g, g[:, ::-1])\n"
                "    v_sym = np.array_equal(g, g[::-1, :])\n"
                "    return {'horizontal': h_sym, 'vertical': v_sym}\n",
                ["symmetry", "analysis"]
            ),
        }
        for name, (code, cats) in builtins.items():
            self.abstractions[name] = {
                "code": code,
                "uses": 0,
                "categories": cats,
            }

    def add_abstraction(self, name: str, code: str, categories: List[str] = None):
        """Add a new abstraction to the library."""
        if name in self.abstractions:
            return  # already exists
        # Validate: the code must be a function definition
        if not re.search(r"^def\s+\w+", code, re.MULTILINE):
            return
        self.abstractions[name] = {
            "code": code,
            "uses": 1,
            "categories": categories or [],
        }

    def extract_from_verified_code(self, code: str) -> int:
        """
        Extract helper functions from verified code and add to library.
        Returns number of new abstractions extracted.
        """
        new_count = 0
        # Find all function definitions except 'transform'
        pattern = re.compile(r"(def\s+(\w+)\s*\([^)]*\)\s*:(?:.*?)(?=\ndef\s|\Z))", re.DOTALL)
        for match in pattern.finditer(code):
            fn_code = match.group(1)
            fn_name = match.group(2)
            if fn_name == "transform":
                continue
            # Check it's not too long (avoid trivial wrappers)
            if 20 < len(fn_code) < 1000:
                self.add_abstraction(fn_name, fn_code)
                new_count += 1
        return new_count

    def get_relevant_abstractions(self, categories: List[str] = None, max_n: int = 5) -> str:
        """
        Get a string of relevant abstractions for inclusion in LLM prompts.
        If categories specified, prioritize those; otherwise return most-used.
        """
        candidates = []
        for name, info in self.abstractions.items():
            if categories:
                overlap = set(info['categories']) & set(categories)
                score = len(overlap) * 10 + info['uses']
            else:
                score = info['uses']
            candidates.append((name, info['code'], score))

        candidates.sort(key=lambda x: -x[2])
        selected = candidates[:max_n]

        if not selected:
            return ""

        lines = ["# Available helper functions (you may use these):", ""]
        for name, code, _ in selected:
            lines.append(f"# {name}:")
            lines.append(code)
            lines.append("")
        return "\n".join(lines)

    def get_stats(self) -> Dict[str, int]:
        return {"total": len(self.abstractions),
                "builtin": sum(1 for v in self.abstractions.values() if v['uses'] == 0),
                "learned": sum(1 for v in self.abstractions.values() if v['uses'] > 0)}


# Global instance
abstraction_lib = AbstractionLibrary()

print(f"[ABSTRACTIONS] Library initialized with {abstraction_lib.get_stats()['builtin']} builtins.")

## LLM Model Loading & Two-Phase Reasoning

In [ ]:
# ============================================================================
# CELL 8: LLM Model Loading (Qwen3-4B, 4-bit Quantized)
# ============================================================================
"""
Lazy-loads a quantized LLM for code generation. Only activates on GPU.
Falls back gracefully when GPU or transformers are not available.
"""

class LLMEngine:
    """
    Manages the LLM lifecycle. Lazy-loads the model on first use.
    Handles 4-bit quantization via bitsandbytes.
    """

    def __init__(self):
        self.model = None
        self.tokenizer = None
        self._loaded = False
        self._load_attempted = False

    def is_available(self) -> bool:
        """Check if LLM can be loaded (GPU + transformers + model access)."""
        if self._load_attempted:
            return self.model is not None
        if not _HAS_TORCH or not _HAS_TRANSFORMERS:
            return False
        try:
            if not torch.cuda.is_available():
                return False
        except Exception:
            return False
        return True

    def load(self):
        """Load the model (called once, cached)."""
        if self._loaded:
            return
        self._load_attempted = True

        print(f"[LLM] Loading {MODEL_NAME}...")
        t0 = time.time()

        try:
            model_kwargs = {
                "torch_dtype": torch.float16,
                "device_map": "auto",
                "trust_remote_code": True,
            }

            if QUANTIZATION_4BIT and _HAS_BNB:
                bnb_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_use_double_quant=True,
                )
                model_kwargs["quantization_config"] = bnb_config
                print("[LLM] Using 4-bit quantization.")

            self.tokenizer = AutoTokenizer.from_pretrained(
                MODEL_NAME, trust_remote_code=True
            )
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token

            self.model = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME, **model_kwargs
            )
            self.model.eval()
            self._loaded = True
            print(f"[LLM] Model loaded in {time.time()-t0:.1f}s")
        except Exception as e:
            print(f"[LLM] Failed to load: {e}")
            self.model = None
            self.tokenizer = None

    def generate(self, messages, temperature=0.3, max_new_tokens=MAX_NEW_TOKENS):
        """Generate text from chat messages. Returns raw text or None."""
        if not self._loaded:
            self.load()
        if self.model is None or self.tokenizer is None:
            return None

        try:
            text = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = self.tokenizer(
                text, return_tensors="pt",
                truncation=True, max_length=MAX_INPUT_TOKENS
            ).to(self.model.device)

            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    top_p=0.92,
                    do_sample=(temperature > 0),
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )

            new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
            return self.tokenizer.decode(new_tokens, skip_special_tokens=True)
        except Exception as e:
            print(f"[LLM] Generation failed: {e}")
            return None


# Global LLM engine instance
llm = LLMEngine()
print(f"[LLM] Engine initialized. Available: {llm.is_available()}")

In [ ]:
# ============================================================================
# CELL 9: Two-Phase LLM Reasoning (Analyze → Code)
# ============================================================================
"""
Two-Phase Reasoning: First analyze the pattern in natural language,
then generate Python code based on the analysis.

This "think first, code second" approach significantly improves accuracy
over direct code generation (from competition research: Jeremy Berman,
Redwood Research).
"""

# === PHASE 1 PROMPT: Natural Language Analysis ===============================

PHASE1_SYSTEM = """You are an expert at analyzing visual pattern puzzles (ARC tasks).
You will see input-output grid pairs. Your job is to:
1. Describe the transformation rule in clear, precise natural language.
2. Identify key features: colors, shapes, objects, spatial relationships.
3. Note any conditional logic (different rules for different inputs).
4. Be specific about dimensions, positions, and color values.

Color reference: 0=black, 1=blue, 2=red, 3=green, 4=yellow,
5=gray, 6=magenta, 7=orange, 8=cyan, 9=brown.

Think step by step. Be thorough and precise."""


PHASE2_SYSTEM = """You are an expert Python programmer who solves ARC puzzles.
You will be given:
1. A natural language analysis of the transformation rule.
2. The input-output grid examples.
3. Helper functions you may use.

Write a single Python function `transform(input_grid)` that implements the rule.

Rules:
- input_grid is a list of lists of ints (each 0-9).
- Return a list of lists of ints.
- Do NOT print anything.
- Only output the function code. No explanation.
- Use numpy as np if helpful.
- Handle variable-sized inputs correctly.
- You may use the provided helper functions."""


def build_enhanced_task_prompt(task: Dict, test_input_idx: int = 0) -> str:
    """
    Build an enhanced prompt with:
    - Natural language grid descriptions (color names, dimensions, objects)
    - Raw grid data
    - Grid feature analysis
    """
    parts = []

    # Training examples with enhanced descriptions
    for i, pair in enumerate(task["train"]):
        inp_desc = describe_grid_natural_language(pair["input"], f"Example {i+1} Input")
        out_desc = describe_grid_natural_language(pair["output"], f"Example {i+1} Output")
        parts.append(inp_desc)
        parts.append(grid_to_str(pair["input"]))
        parts.append(out_desc)
        parts.append(grid_to_str(pair["output"]))

        # Dimension analysis
        inp_s = grid_shape(pair["input"])
        out_s = grid_shape(pair["output"])
        if inp_s != out_s:
            parts.append(f"Size change: {inp_s[0]}x{inp_s[1]} → {out_s[0]}x{out_s[1]}")
        parts.append("")

    # Test input
    test_input = task["test"][test_input_idx]["input"]
    parts.append(describe_grid_natural_language(test_input, "Test Input"))
    parts.append(grid_to_str(test_input))

    return "\n".join(parts)


def two_phase_llm_solve(
    task: Dict,
    engine: LLMEngine,
    test_input_idx: int = 0,
    temperatures: List[float] = [0.2, 0.4, 0.6, 0.8],
    max_time: float = MAX_LLM_TIME_SECONDS,
) -> List[Tuple[str, float, int]]:
    """
    Two-phase LLM solving:
    Phase 1: Analyze pattern in natural language
    Phase 2: Generate Python code based on analysis

    Returns list of (code, score, num_passed) tuples.
    """
    if not engine.is_available():
        return []

    train_pairs = task["train"]
    task_prompt = build_enhanced_task_prompt(task, test_input_idx)
    results = []
    t_start = time.time()

    for temp in temperatures:
        if time.time() - t_start > max_time:
            break

        # --- Phase 1: Analysis ---
        analysis_messages = [
            {"role": "system", "content": PHASE1_SYSTEM},
            {"role": "user", "content": f"Analyze this ARC task:\n\n{task_prompt}\n\n"
             f"Describe the transformation rule precisely."},
        ]
        analysis = engine.generate(analysis_messages, temperature=temp,
                                   max_new_tokens=1024)
        if not analysis:
            continue

        # --- Phase 2: Code Generation ---
        helper_code = abstraction_lib.get_relevant_abstractions(max_n=3)
        code_messages = [
            {"role": "system", "content": PHASE2_SYSTEM},
            {"role": "user", "content":
             f"## Pattern Analysis\n{analysis}\n\n"
             f"## Examples\n{task_prompt}\n\n"
             f"{helper_code}\n"
             f"Write a transform(input_grid) function."},
        ]
        raw_code = engine.generate(code_messages, temperature=temp,
                                    max_new_tokens=MAX_NEW_TOKENS)
        if not raw_code:
            continue

        code = extract_function_code(raw_code)
        if not code:
            continue

        # Verify with partial match scoring
        score, passed, total = verify_code_on_training(code, train_pairs)
        if score > 0.0:
            results.append((code, score, passed))

    # Also try direct code generation (no phase 1) for comparison
    if time.time() - t_start < max_time * 0.5:
        direct_messages = [
            {"role": "system", "content": PHASE2_SYSTEM},
            {"role": "user", "content":
             f"{helper_code}\n\n"
             f"{task_prompt}\n\n"
             f"Write a transform(input_grid) function."},
        ]
        for temp in [0.2, 0.5]:
            raw = engine.generate(direct_messages, temperature=temp)
            if raw:
                code = extract_function_code(raw)
                if code:
                    score, passed, total = verify_code_on_training(code, train_pairs)
                    if score > 0.0:
                        results.append((code, score, passed))

    # Deduplicate by code hash, keep best score
    seen = {}
    for code, score, passed in results:
        h = hashlib.md5(code.encode()).hexdigest()[:12]
        if h not in seen or score > seen[h][1]:
            seen[h] = (code, score, passed)

    return [(c, s, p) for c, s, p in seen.values()]


print("[TWO-PHASE] Two-phase LLM reasoning ready.")

## Evolutionary Program Synthesis

In [ ]:
# ============================================================================
# CELL 10: Evolutionary Program Synthesis
# ============================================================================
"""
After LLM generates initial code candidates, mutate and cross-over programs
to explore the solution space more broadly. Inspired by DreamCoder and
Eric Pang's SOAR framework.

Mutation operators:
  - Change numeric constants (thresholds, indices, offsets)
  - Swap operations (flip_h ↔ flip_v, rotate90 ↔ rotate270)
  - Add/remove conditional branches
  - Replace color values

Crossover:
  - Combine helper functions from two programs
  - Swap transform bodies between verified programs
"""

class ProgramMutator:
    """
    Mutates and cross-breeds Python code to generate new candidates.
    All mutations are verified against training pairs.
    """

    # Common numeric substitutions
    NUMERIC_MUTATIONS = {
        '0': ['1', '0', '-1'],
        '1': ['0', '2', '1', '-1'],
        '2': ['1', '3', '2', '0'],
        '3': ['2', '4', '3', '1'],
        '4': ['3', '5', '4', '2'],
        '5': ['4', '6', '5', '3'],
        '6': ['5', '7', '6', '4'],
        '7': ['6', '8', '7', '5'],
        '8': ['7', '9', '8', '6'],
        '9': ['8', '0', '9', '7'],
    }

    # Operation swap patterns
    OP_SWAPS = [
        ("[:, ::-1]", "[::-1, :]"),   # flip_h ↔ flip_v
        ("[::-1, :]", "[:, ::-1]"),    # flip_v ↔ flip_h
        ("rot90(g, 1)", "rot90(g, 3)"),  # rotate90 ↔ rotate270
        ("rot90(g, 3)", "rot90(g, 1)"),  # rotate270 ↔ rotate90
        ("rot90(g, 2)", "g"),          # rotate180 ↔ identity
        (".T", ""),                   # transpose ↔ identity
        ("(g + 1)", "(g - 1)"),      # add 1 ↔ sub 1
        ("(g - 1)", "(g + 1)"),      # sub 1 ↔ add 1
        ("% 10", ""),                 # mod 10 ↔ no mod
        ("> 0", "== 0"),             # nonzero ↔ zero
        ("== 0", "> 0"),             # zero ↔ nonzero
        ("any(axis=1)", "any(axis=0)"),  # row-any ↔ col-any
        ("any(axis=0)", "any(axis=1)"),  # col-any ↔ row-any
        ("axis=0", "axis=1"),
        ("axis=1", "axis=0"),
    ]

    def mutate_numeric(self, code: str) -> List[str]:
        """Generate mutations by changing numeric constants."""
        mutations = []
        for original, replacements in self.NUMERIC_MUTATIONS.items():
            for replacement in replacements:
                if replacement == original:
                    continue
                # Only replace standalone numbers (not in variable names)
                import re as _re
                pattern = _re.compile(r'(?<![\w])' + _re.escape(original) + r'(?![\w])')
                mutated = pattern.sub(replacement, code)
                if mutated != code:
                    mutations.append(mutated)
        return mutations

    def mutate_operations(self, code: str) -> List[str]:
        """Generate mutations by swapping operations."""
        mutations = []
        for old_op, new_op in self.OP_SWAPS:
            if old_op in code:
                mutated = code.replace(old_op, new_op, 1)
                mutations.append(mutated)
        return mutations

    def mutate_color_value(self, code: str) -> List[str]:
        """Mutate color constants (e.g., change == 1 to == 2)."""
        mutations = []
        for c in range(10):
            for c2 in range(10):
                if c == c2:
                    continue
                # Replace color == c with color == c2
                import re as _re
                for pattern_str in [f"== {c}", f"!= {c}", f"== {c}", f"({c})",
                                   f"[{c}]", f"= {c}"]:
                    if pattern_str in code:
                        mutated = code.replace(pattern_str, pattern_str.replace(str(c), str(c2)), 1)
                        if mutated != code:
                            mutations.append(mutated)
            if len(mutations) >= 5:  # limit
                break
        return mutations[:5]

    def mutate_indentation_logic(self, code: str) -> List[str]:
        """Generate mutations that swap if/else branches."""
        mutations = []
        # Try swapping True/False conditions
        if "if " in code:
            # Change > to < and vice versa
            mutations.append(code.replace(">", "<", 1))
            mutations.append(code.replace("<", ">", 1))
            mutations.append(code.replace(">=", "<=", 1))
            mutations.append(code.replace("<=", ">=", 1))
        return [m for m in mutations if m != code][:3]

    def crossover(self, code1: str, code2: str) -> List[str]:
        """
        Cross-over two programs: take helper functions from one and
        transform body from the other.
        """
        children = []

        # Extract helper functions from both
        def extract_helpers(code):
            helpers = []
            pattern = re.compile(r"(def\s+(\w+)\s*\([^)]*\)\s*:.*?)(?=\ndef\s+transform|$)",
                                re.DOTALL)
            for m in pattern.finditer(code):
                if m.group(2) != "transform":
                    helpers.append(m.group(1))
            return helpers

        def extract_transform(code):
            pattern = re.compile(r"(def\s+transform\s*\([^)]*\)\s*:.*?$)", re.DOTALL)
            m = pattern.search(code)
            return m.group(1) if m else None

        helpers1 = extract_helpers(code1)
        helpers2 = extract_helpers(code2)
        transform1 = extract_transform(code1)
        transform2 = extract_transform(code2)

        if transform1 and helpers2:
            children.append("\n".join(helpers2) + "\n" + transform1)
        if transform2 and helpers1:
            children.append("\n".join(helpers1) + "\n" + transform2)

        return children


def evolutionary_synthesis(
    initial_codes: List[Tuple[str, float, int]],
    train_pairs: List[Dict],
    population_size: int = EVOLUTIONARY_POPULATION,
    generations: int = EVOLUTIONARY_GENERATIONS,
) -> List[Tuple[str, float, int]]:
    """
    Evolve an initial set of code candidates using mutation and crossover.

    Args:
        initial_codes: List of (code, score, num_passed) from LLM
        train_pairs: Training pairs for verification
        population_size: Max candidates in the population
        generations: Number of evolution rounds

    Returns:
        List of (code, score, num_passed) sorted by score desc
    """
    if not initial_codes:
        return []

    mutator = ProgramMutator()
    population = {}  # code_hash → (code, score, passed)

    # Seed population with initial candidates
    for code, score, passed in initial_codes:
        h = hashlib.md5(code.encode()).hexdigest()[:12]
        population[h] = (code, score, passed)

    for gen in range(generations):
        new_candidates = []

        # Get current best codes
        current_codes = [c for c, s, p in population.values()]

        for code, score, passed in list(population.values()):
            # Mutate only if score < 1.0 (already perfect? no need)
            if score >= 1.0:
                continue

            # Numeric mutations
            for mutant in mutator.mutate_numeric(code)[:2]:
                new_candidates.append(mutant)

            # Operation swaps
            for mutant in mutator.mutate_operations(code)[:2]:
                new_candidates.append(mutant)

            # Color mutations
            for mutant in mutator.mutate_color_value(code)[:1]:
                new_candidates.append(mutant)

            # Logic mutations
            for mutant in mutator.mutate_indentation_logic(code)[:1]:
                new_candidates.append(mutant)

        # Crossover between top candidates
        top_codes = [c for c, s, p in sorted(population.values(),
                                               key=lambda x: -x[1])][:5]
        for i in range(len(top_codes)):
            for j in range(i+1, len(top_codes)):
                children = mutator.crossover(top_codes[i], top_codes[j])
                new_candidates.extend(children)

        # Verify new candidates
        for candidate in new_candidates:
            h = hashlib.md5(candidate.encode()).hexdigest()[:12]
            if h in population:
                continue
            score, passed, total = verify_code_on_training(candidate, train_pairs)
            if score > 0.0:
                population[h] = (candidate, score, passed)

        # Prune to population_size (keep best)
        if len(population) > population_size:
            sorted_items = sorted(population.items(), key=lambda x: -x[1][1])
            population = dict(sorted_items[:population_size])

        # Early termination if we have a perfect solution
        if any(s >= 1.0 for s, p in [(v[1], v[2]) for v in population.values()]):
            break

    results = sorted(population.values(), key=lambda x: (-x[1], -x[2]))
    return results


print("[EVOLUTION] Program synthesis engine ready.")

## Multi-Strategy Ensemble Solver

In [ ]:
# ============================================================================
# CELL 11: Multi-Strategy Ensemble Solver
# ============================================================================
"""
Orchestrates all solving strategies:
1. Fast heuristic first-pass (all 90+ solvers)
2. LLM two-phase reasoning (if heuristics don't achieve perfect score)
3. Evolutionary synthesis of LLM candidates
4. Ensemble: pick top-2 distinct predictions by confidence

The ensemble uses partial-match scoring to rank candidates, and selects
the top 2 distinct outputs for submission (either correct = full credit).
"""

def _apply_fn_safe(fn, grid) -> Optional[List[List[int]]]:
    """Apply a function safely, returning normalized grid or None."""
    try:
        result = fn(copy.deepcopy(grid))
        return normalize_grid(result)
    except Exception:
        return None


def _zero_grid_fallback(input_grid: List[List[int]]) -> List[List[int]]:
    """Last-resort fallback: return same-size zero grid."""
    rows = len(input_grid)
    cols = len(input_grid[0]) if input_grid else 0
    return [[0] * cols for _ in range(rows)]


def solve_task(
    task: Dict,
    engine: LLMEngine,
    test_input_idx: int = 0,
    task_id: str = "",
) -> Tuple[List[List[List[int]]], Dict[str, Any]]:
    """
    Solve a single task using the full ensemble pipeline.

    Returns:
        (candidates, stats) where candidates is a list of output grids
        and stats contains solving metadata.
    """
    stats = {
        "task_id": task_id,
        "heuristic_best_score": 0.0,
        "llm_candidates": 0,
        "evolved_candidates": 0,
        "total_candidates": 0,
        "strategy": "none",
    }

    train_pairs = task["train"]
    test_input = task["test"][test_input_idx]["input"]
    all_candidates: List[Tuple[List[List[int]], float, str]] = []
    # (grid, score, source_label)

    # =========================================================================
    # PASS 1: Fast Heuristic Solvers
    # =========================================================================
    best_heuristics = find_best_heuristics(train_pairs, min_score=1.0, max_results=5)

    if best_heuristics:
        best_score = best_heuristics[0][2]
        stats["heuristic_best_score"] = best_score
        stats["strategy"] = "heuristic"

        for name, fn, score, passed, total in best_heuristics:
            result = _apply_fn_safe(fn, test_input)
            if result is not None:
                all_candidates.append((result, score, f"heuristic:{name}"))

        # If we found perfect heuristic solutions, add partial-match ones too
        partial = find_best_partial_heuristics(train_pairs, max_results=3)
        for name, fn, score, passed, total in partial:
            if score < 1.0:  # only add imperfect ones (perfect already added)
                result = _apply_fn_safe(fn, test_input)
                if result is not None:
                    all_candidates.append((result, score, f"heuristic_partial:{name}"))

    # =========================================================================
    # PASS 2: LLM Two-Phase Reasoning (if heuristic score < 1.0 or to add more)
    # =========================================================================
    llm_codes = []
    if engine.is_available():
        # Always try LLM for additional candidates (even if heuristics passed)
        # Use shorter time budget if heuristics already solved
        budget = MAX_LLM_TIME_SECONDS if stats["heuristic_best_score"] < 1.0 else 60
        llm_codes = two_phase_llm_solve(task, engine, test_input_idx,
                                         max_time=budget)
        stats["llm_candidates"] = len(llm_codes)

        # Run LLM codes on test input
        for code, score, passed in llm_codes:
            result = safe_execute_code(code, test_input)
            if result is not None:
                all_candidates.append((result, score, "llm"))

                # Extract abstractions from high-scoring code
                if score >= 0.8:
                    abstraction_lib.extract_from_verified_code(code)

    # =========================================================================
    # PASS 3: Evolutionary Synthesis
    # =========================================================================
    if llm_codes:
        evolved = evolutionary_synthesis(llm_codes, train_pairs)
        stats["evolved_candidates"] = len(evolved)

        # Run evolved codes on test input
        for code, score, passed in evolved:
            # Skip if we already have this code's output
            h = hashlib.md5(code.encode()).hexdigest()[:12]
            result = safe_execute_code(code, test_input)
            if result is not None:
                all_candidates.append((result, score, "evolved"))

    # =========================================================================
    # ENSEMBLE: Select Top-2 Distinct Candidates
    # =========================================================================
    # Sort by score descending, then deduplicate by grid hash
    all_candidates.sort(key=lambda x: -x[1])

    selected = []
    seen_hashes = set()
    for grid, score, source in all_candidates:
        h = grid_hash(grid)
        if h not in seen_hashes:
            seen_hashes.add(h)
            selected.append(grid)
            if len(selected) >= NUM_SUBMISSION_ATTEMPTS:
                break

    # Fill remaining slots with fallback
    while len(selected) < NUM_SUBMISSION_ATTEMPTS:
        if best_heuristics:
            # Use best heuristic even if we already included it
            result = _apply_fn_safe(best_heuristics[0][1], test_input)
            if result is not None:
                h = grid_hash(result)
                if h not in seen_hashes:
                    seen_hashes.add(h)
                    selected.append(result)
                    continue
        selected.append(_zero_grid_fallback(test_input))
        break

    stats["total_candidates"] = len(all_candidates)
    if stats["heuristic_best_score"] >= 1.0:
        stats["strategy"] = "heuristic"
    elif any(s >= 1.0 for _, s, _ in all_candidates):
        stats["strategy"] = "llm_or_evolved"
    elif all_candidates:
        stats["strategy"] = "partial_match"
    else:
        stats["strategy"] = "zero_fallback"

    return selected[:NUM_SUBMISSION_ATTEMPTS], stats


print("[ENSEMBLE] Multi-strategy solver ready.")

## Main Pipeline & Submission

In [ ]:
# ============================================================================
# CELL 12: Main Solving Pipeline
# ============================================================================
"""
Main pipeline that orchestrates solving all evaluation tasks.
Includes progress tracking, statistics, and time budgeting.
"""

def solve_all_tasks(
    eval_tasks: Dict[str, Dict],
    engine: LLMEngine,
    max_tasks: Optional[int] = None,
    total_time_budget: float = 9 * 3600 - 600,  # 9 hours minus 10 min margin
) -> Tuple[Dict[str, List], Dict[str, Any]]:
    """
    Solve all evaluation tasks with time budgeting.

    Returns:
        (submission_dict, global_stats)
    """
    submission: Dict[str, List] = {}
    task_ids = list(eval_tasks.keys())
    if max_tasks is not None:
        task_ids = task_ids[:max_tasks]

    total = len(task_ids)
    per_task_budget = total_time_budget / max(total, 1)

    print(f"\n{'='*60}")
    print(f"SOLVING {total} evaluation tasks")
    print(f"Per-task budget: {per_task_budget:.0f}s | "
          f"Total budget: {total_time_budget/3600:.1f}h")
    print(f"{'='*60}\n")

    global_stats = {
        "heuristic": 0, "llm": 0, "evolved": 0,
        "partial_match": 0, "zero_fallback": 0,
        "total": total, "solved": 0,
    }

    overall_start = time.time()

    for idx, task_id in enumerate(tqdm(task_ids, desc="Solving tasks")):
        task_start = time.time()
        task = eval_tasks[task_id]

        candidates, task_stats = solve_task(task, engine, 0, task_id)

        # Handle multiple test inputs per task
        num_test = len(task["test"])
        if num_test > 1:
            # For tasks with multiple test inputs, solve each one
            all_attempts = []
            for test_idx in range(num_test):
                cands, _ = solve_task(task, engine, test_idx, task_id)
                all_attempts.extend(cands)
            submission[task_id] = all_attempts[:NUM_SUBMISSION_ATTEMPTS * num_test]
        else:
            submission[task_id] = candidates

        # Track statistics
        strategy = task_stats["strategy"]
        if strategy == "heuristic":
            global_stats["heuristic"] += 1
            global_stats["solved"] += 1
        elif strategy in ("llm_or_evolved", "evolved"):
            global_stats["llm"] += 1
            global_stats["solved"] += 1
        elif strategy == "partial_match":
            global_stats["partial_match"] += 1
        else:
            global_stats["zero_fallback"] += 1

        task_elapsed = time.time() - task_start

        # Progress report every 20 tasks
        if (idx + 1) % 20 == 0 or idx == total - 1:
            elapsed_total = time.time() - overall_start
            rate = (idx + 1) / max(elapsed_total, 1)
            remaining = (total - idx - 1) / max(rate, 0.001)
            print(f"\n  [{idx+1}/{total}] "
                  f"Heuristic: {global_stats['heuristic']} | "
                  f"LLM: {global_stats['llm']} | "
                  f"Partial: {global_stats['partial_match']} | "
                  f"Zero: {global_stats['zero_fallback']} | "
                  f"Rate: {rate:.2f} tasks/s | "
                  f"ETA: {remaining/60:.0f}min | "
                  f"Abstractions: {abstraction_lib.get_stats()['total']}")

    total_elapsed = time.time() - overall_start
    global_stats["elapsed_seconds"] = total_elapsed

    print(f"\n{'='*60}")
    print(f"PIPELINE COMPLETE in {total_elapsed/60:.1f} minutes")
    print(f"  Heuristic:  {global_stats['heuristic']}/{total}")
    print(f"  LLM/Evolved: {global_stats['llm']}/{total}")
    print(f"  Partial:    {global_stats['partial_match']}/{total}")
    print(f"  Zero:       {global_stats['zero_fallback']}/{total}")
    print(f"  Abstractions learned: {abstraction_lib.get_stats()['learned']}")
    print(f"{'='*60}\n")

    return submission, global_stats


def save_submission(submission: Dict, path: str) -> None:
    """Write submission.json to disk."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(submission, f)
    print(f"[SUBMISSION] Saved to {path}")
    print(f"[SUBMISSION] Tasks: {len(submission)}")
    total_attempts = sum(len(v) for v in submission.values())
    print(f"[SUBMISSION] Total attempts: {total_attempts}")


print("[PIPELINE] Main solving pipeline ready.")

## Training Validation (Self-Scoring)

In [ ]:
# ============================================================================
# CELL 13: Training Validation (Score on Training Tasks)
# ============================================================================
"""
Validate the solver on training tasks where ground truth is available.
This gives an estimated accuracy before running on evaluation tasks.
"""

def validate_on_training(
    train_tasks: Dict[str, Dict],
    engine: LLMEngine,
    max_tasks: int = 30,
) -> Dict[str, Any]:
    """
    Score the solver on training tasks (which include test outputs).
    Returns accuracy statistics.
    """
    print(f"\n{'='*60}")
    print(f"VALIDATION on {max_tasks} training tasks")
    print(f"{'='*60}\n")

    task_ids = list(train_tasks.keys())[:max_tasks]
    total_pairs = 0
    correct_pairs = 0
    per_task = {}

    for task_id in tqdm(task_ids, desc="Validating"):
        task = train_tasks[task_id]
        num_test = len(task["test"])
        task_correct = 0

        for test_idx in range(num_test):
            total_pairs += 1
            candidates, stats = solve_task(task, engine, test_idx, task_id)

            if "output" in task["test"][test_idx]:
                expected = task["test"][test_idx]["output"]
                for candidate in candidates:
                    if grids_equal(candidate, expected):
                        correct_pairs += 1
                        task_correct += 1
                        break

        per_task[task_id] = task_correct

    accuracy = correct_pairs / max(total_pairs, 1)
    tasks_with_correct = sum(1 for v in per_task.values() if v > 0)

    print(f"\n  Validation Results:")
    print(f"    Pairs correct: {correct_pairs}/{total_pairs} ({accuracy:.1%})")
    print(f"    Tasks with ≥1 correct: {tasks_with_correct}/{len(task_ids)}")
    print(f"    Abstractions: {abstraction_lib.get_stats()}")

    # Show per-task breakdown
    print(f"\n  Per-task (first 15):")
    for tid in list(task_ids)[:15]:
        n_test = len(train_tasks[tid]["test"])
        print(f"    {tid}: {per_task[tid]}/{n_test}")

    return {
        "accuracy": accuracy,
        "correct_pairs": correct_pairs,
        "total_pairs": total_pairs,
        "tasks_with_correct": tasks_with_correct,
        "per_task": per_task,
    }


print("[VALIDATION] Training validation ready.")

## Execute: Run the Full Pipeline

In [ ]:
# ============================================================================
# CELL 14: Main Entry Point — Run the Full Pipeline
# ============================================================================
"""
This is the main execution cell. Run all cells above, then this cell.
It loads data, optionally validates on training, solves evaluation tasks,
and saves submission.json.
"""

def main():
    """Main entry point."""
    print("=" * 60)
    print("ARC PRIZE 2026 — ARC-AGI-2 SOLVER v2")
    print("Evolutionary + Two-Phase LLM + Ensemble")
    print("=" * 60)
    print(f"Python: {sys.version}")
    if _HAS_TORCH:
        print(f"PyTorch: {torch.__version__}")
        print(f"CUDA: {torch.cuda.is_available()}")
        if torch.cuda.is_available():
            print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"NumPy: {np.__version__}")
    print(f"Heuristics: {len(ALL_HEURISTICS)}")
    print()

    # --- Step 1: Load data ---
    print("[STEP 1/4] Loading ARC data...")
    eval_tasks, train_tasks = load_arc_data(DATA_DIR)

    if not eval_tasks:
        print("[ERROR] No evaluation tasks found!")
        print(f"  DATA_DIR: {DATA_DIR}")
        if os.path.exists(DATA_DIR):
            print(f"  Contents: {os.listdir(DATA_DIR)}")
        return

    # Show sample tasks
    print("\n  Sample tasks:")
    for tid in list(eval_tasks.keys())[:3]:
        print_task_summary(eval_tasks[tid], tid)

    # --- Step 2: Load LLM (if available) ---
    print("\n[STEP 2/4] Loading LLM model...")
    if llm.is_available():
        llm.load()
    else:
        print("  [SKIP] LLM not available (no GPU/transformers). Using heuristics only.")

    # --- Step 3: Validate on training (quick check) ---
    if train_tasks:
        print("\n[STEP 3/4] Quick validation on training tasks...")
        val_n = min(15, len(train_tasks))
        val_results = validate_on_training(train_tasks, llm, max_tasks=val_n)
    else:
        print("\n[STEP 3/4] No training tasks available, skipping validation.")

    # --- Step 4: Solve evaluation tasks ---
    print("\n[STEP 4/4] Solving evaluation tasks...")
    submission, global_stats = solve_all_tasks(eval_tasks, llm)

    # --- Save submission ---
    save_submission(submission, OUTPUT_PATH)

    # --- Final summary ---
    print("\n" + "=" * 60)
    print("FINAL SUMMARY")
    print("=" * 60)
    print(f"  Total tasks: {global_stats['total']}")
    print(f"  Heuristic solved: {global_stats['heuristic']}")
    print(f"  LLM/Evolved solved: {global_stats['llm']}")
    print(f"  Partial match: {global_stats['partial_match']}")
    print(f"  Zero fallback: {global_stats['zero_fallback']}")
    if 'accuracy' in dir() and 'val_results' in dir():
        print(f"  Training accuracy: {val_results['accuracy']:.1%}")
    print(f"  Total time: {global_stats.get('elapsed_seconds', 0)/60:.1f} min")
    print(f"  Abstractions: {abstraction_lib.get_stats()}")
    print(f"  Output: {OUTPUT_PATH}")
    print("=" * 60)


# Run the main pipeline
if __name__ == "__main__":
    main()
else:
    # In Kaggle notebook, just call main() directly
    main()

## Submission Verification

In [ ]:
# ============================================================================
# CELL 15: Submission Verification & Sanity Checks
# ============================================================================
"""
Verify the submission file is well-formed before submitting.
Checks: format, grid dimensions, value ranges, task coverage.
"""

def verify_submission(submission_path: str, eval_tasks: Dict[str, Dict]) -> Dict[str, Any]:
    """
    Run sanity checks on the submission file.
    Returns a dict with check results.
    """
    if not os.path.exists(submission_path):
        return {"error": "Submission file not found"}

    with open(submission_path, "r") as f:
        submission = json.load(f)

    checks = {
        "file_exists": True,
        "is_valid_json": True,
        "tasks_in_submission": len(submission),
        "tasks_expected": len(eval_tasks),
        "all_tasks_covered": set(submission.keys()) == set(eval_tasks.keys()),
        "missing_tasks": sorted(set(eval_tasks.keys()) - set(submission.keys())),
        "extra_tasks": sorted(set(submission.keys()) - set(eval_tasks.keys())),
        "format_errors": [],
        "value_errors": [],
    }

    for task_id, attempts in submission.items():
        # Check it's a list
        if not isinstance(attempts, list):
            checks["format_errors"].append(f"{task_id}: not a list")
            continue

        # Check exactly 2 attempts per test input
        expected_test = len(eval_tasks.get(task_id, {}).get("test", []))
        expected_attempts = expected_test * 2
        if len(attempts) != expected_attempts and expected_test > 0:
            checks["format_errors"].append(
                f"{task_id}: {len(attempts)} attempts, expected {expected_attempts}")

        for attempt_idx, grid in enumerate(attempts):
            if not isinstance(grid, list):
                checks["format_errors"].append(
                    f"{task_id}[{attempt_idx}]: not a list")
                continue
            if not all(isinstance(row, list) for row in grid):
                checks["format_errors"].append(
                    f"{task_id}[{attempt_idx}]: not a list of lists")
                continue
            for r, row in enumerate(grid):
                for c, val in enumerate(row):
                    if not isinstance(val, int) or val < 0 or val > 9:
                        checks["value_errors"].append(
                            f"{task_id}[{attempt_idx}][{r}][{c}]={val} (must be 0-9 int)")

    checks["num_format_errors"] = len(checks["format_errors"])
    checks["num_value_errors"] = len(checks["value_errors"])
    checks["is_valid"] = (
        checks["all_tasks_covered"]
        and checks["num_format_errors"] == 0
        and checks["num_value_errors"] == 0
    )

    return checks


# Run verification
print("[VERIFY] Running submission sanity checks...")
try:
    eval_tasks_check, _ = load_arc_data(DATA_DIR)
    results = verify_submission(OUTPUT_PATH, eval_tasks_check)
    print(f"\n  File exists: {results.get('file_exists')}")
    print(f"  Valid JSON: {results.get('is_valid_json')}")
    print(f"  Tasks: {results.get('tasks_in_submission')}/{results.get('tasks_expected')}")
    print(f"  All covered: {results.get('all_tasks_covered')}")
    print(f"  Format errors: {results.get('num_format_errors')}")
    print(f"  Value errors: {results.get('num_value_errors')}")
    print(f"  IS VALID: {results.get('is_valid')}")

    if results.get("format_errors"):
        print("\n  Format errors (first 5):")
        for err in results["format_errors"][:5]:
            print(f"    - {err}")

    if results.get("value_errors"):
        print("\n  Value errors (first 5):")
        for err in results["value_errors"][:5]:
            print(f"    - {err}")

    if results.get("is_valid"):
        print("\n  ✅ Submission looks valid! Ready to submit.")
    else:
        print("\n  ❌ Submission has errors. Please fix before submitting.")
except Exception as e:
    print(f"[VERIFY] Could not verify: {e}")